# Import data

In [18]:
import pandas as pd
import re

# Visit https://www.tennisabstract.com/charting/20120129-M-Australian_Open-F-Novak_Djokovic-Rafael_Nadal.html
#     for insteresting stats
match_id = "20120129-M-Australian_Open-F-Novak_Djokovic-Rafael_Nadal"

dfAll = pd.read_csv("./data/charting-m-points-2010s.csv")

df = dfAll[dfAll["match_id"] == match_id].copy()
df = df.sort_values("Pt").set_index("Pt")

df["Gm2"] = df["Gm2"].astype(int)

df.head()

,match_id,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
Pt,,,,,,,,,,,,,
1,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,0-0,1,True,1,6w,4b27b3f3f1b2f3f1d@,NaN,1
2,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,15-0,1,True,1,6w,4f27b3n@,NaN,2
3,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,15-15,1,True,1,6w,4b28f3f3b3f3b3f2f3f1f1b2f1n@,NaN,2
4,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,15-30,1,True,1,6b28f3*,NaN,NaN,1
5,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,30-30,1,True,1,4w,5b39b1b3b3f1f2f2d@,NaN,1


In [61]:
match_id_gs = "2012-ausopen-1701"

dfGSAll = pd.read_csv("./data/tennisSlamPointByPoint/2012-ausopen-points.csv")

dfGS = dfGSAll[dfGSAll["match_id"] == match_id_gs].copy()
dfGS = dfGS.sort_values("PointNumber").set_index("PointNumber")
dfGS.index.name = "Pt"
#dfGS.index = range(1, len(dfGS) + 1)

print(len(dfGS) == len(df))
dfGS.head()

True


,match_id,ElapsedTime,SetNo,P1GamesWon,P2GamesWon,SetWinner,GameNo,GameWinner,PointWinner,PointServer,...,Speed_MPH,P1BreakPointMissed,P2BreakPointMissed,ServeIndicator,Serve_Direction,Winner_FH,Winner_BH,ServingTo,P1TurningPoint,P2TurningPoint
Pt,,,,,,,,,,,,,,,,,,,,,
1,2012-ausopen-1701,0:00:00,1,0,0,0,1,0,1,1,...,113,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
2,2012-ausopen-1701,0:00:36,1,0,0,0,1,0,2,1,...,88,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-ausopen-1701,0:01:18,1,0,0,0,1,0,2,1,...,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
4,2012-ausopen-1701,0:02:21,1,0,0,0,1,0,1,1,...,124,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN
5,2012-ausopen-1701,0:02:51,1,0,0,0,1,0,1,1,...,93,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN


# Verifying tactics
Let us verify if my perceived tactics of both players are correct.
1. Djokovic: "My backhand is better than yours."
    Djokovic's main tactic was to send a deep backhand down the line (from the ad-court) to Nadal's backhand (on the deuce-court).
2. Nadal: "My forehand is monstruous."
    Nadal, on the other hand, was trying to send his forhand inside-out crosscourt (from the deuce-court) onto Djokovic's backhand to create angles.

## Separate rally from serve

In [2]:
players = {
    1: "Novak Djokovic",   # server code 1
    2: "Rafael Nadal"
}
handed = {
    "Novak Djokovic": "R",
    "Rafael Nadal": "L"
}

SHOT_LETTERS = set("fbrsvzopuylmhijktq")

# Split each point into serve code and rally code
def SplitServeFromRally(row):
    sequence = row["2nd"] if isinstance(row["2nd"], str) and row["2nd"].strip() else row["1st"]
    # Handle missing data
    if not isinstance(sequence, str):
        return "", ""
    for idx, char in enumerate(sequence):
        if char.lower() in SHOT_LETTERS:
            return sequence[:idx], sequence[idx:]
    # No rally shot found -> serve-only point (ace, double fault, unreturnable, etc.)
    return sequence, ""
    
df[["serve", "rally"]] = df.apply(SplitServeFromRally, axis=1, result_type="expand")

df[['rally']].head()

,rally
Pt,
1,b27b3f3f1b2f3f1d@
2,f27b3n@
3,b28f3f3b3f3b3f2f3f1f1b2f1n@
4,b28f3*
5,b39b1b3b3f1f2f2d@


## Separate shots

In [3]:
POSITION_FLAGS = "+-^=;"  # approach, net, baseline smash, drop-volley, net-cord markers

def ParseToken(token: str) -> dict:
    shot = token[0].lower()
    rest = token[1:]

    positions = ""
    # Handle cases like b+;3 (approach + net-cord)
    while rest and rest[0] in POSITION_FLAGS:
        positions += rest[0]
        rest = rest[1:]
    positions = positions if positions else pd.NA

    direction = int(rest[0]) if rest and rest[0] in "0123" else None
    if direction is not None:
        rest = rest[1:]

    error_code = rest[0] if rest and rest[0] in "nwdxge!" else None
    if error_code:
        rest = rest[1:]

    outcome = rest[0] if rest and rest[0] in "*@#" else None

    return {
        "stroke": shot,
        "positions": positions,   # e.g., "+", "-", etc.
        "direction": direction,   # 1/2/3 (0 or None if missing)
        "errorCode": error_code,
        "outcome": outcome,
    }

In [4]:
# Tokens: each shot starts with a letter and consumes all characters until the next letter
TOKENS_REGEX = re.compile(r"[A-Za-z][^A-Za-z]*")

# Iterate through all points
shotRows = []

for pt, row in df.iterrows():
    rally = row["rally"]
    if not rally:
        continue  # ace, double fault, etc.

    tokens = TOKENS_REGEX.findall(rally)
    server = players[row["Svr"]]
    returner = players[1 if row["Svr"] == 2 else 2]

    for idx, token in enumerate(tokens):
        hitter = returner if idx % 2 == 0 else server
        parsed = ParseToken(token)
        shotRows.append({
            "Pt": pt,
            "shotIndex": idx,
            "server": server,
            "hitter": hitter,
            **parsed
        })

shots = pd.DataFrame(shotRows)
shots['direction'] = shots['direction'].astype("Int64")

In [5]:
def direction_label(direction, hitter):
    if direction is None:
        return "unknown"
    opponent = players[1] if hitter == players[2] else players[2]
    if handed[opponent] == "R":
        mapping = {1: "to opponent FH", 2: "middle", 3: "to opponent BH"}
    else:
        mapping = {1: "to opponent BH", 2: "middle", 3: "to opponent FH"}
    return mapping.get(direction, pd.NA)



shots["targetZone"] = shots.apply(
    lambda r: direction_label(r["direction"], r["hitter"]),
    axis=1
)

In [6]:
shots[shots['direction'].notna()]

,Pt,shotIndex,server,hitter,stroke,positions,direction,errorCode,outcome,targetZone
0,1,0,Novak Djokovic,Rafael Nadal,b,NaN,2,NaN,NaN,middle
1,1,1,Novak Djokovic,Novak Djokovic,b,NaN,3,NaN,NaN,to opponent FH
2,1,2,Novak Djokovic,Rafael Nadal,f,NaN,3,NaN,NaN,to opponent BH
3,1,3,Novak Djokovic,Novak Djokovic,f,NaN,1,NaN,NaN,to opponent BH
4,1,4,Novak Djokovic,Rafael Nadal,b,NaN,2,NaN,NaN,middle
...,...,...,...,...,...,...,...,...,...,...
2028,368,4,Novak Djokovic,Rafael Nadal,b,NaN,1,NaN,NaN,to opponent FH
2029,368,5,Novak Djokovic,Novak Djokovic,f,NaN,1,NaN,NaN,to opponent BH
2030,368,6,Novak Djokovic,Rafael Nadal,b,NaN,1,NaN,NaN,to opponent FH
2032,369,0,Novak Djokovic,Rafael Nadal,s,NaN,2,NaN,NaN,middle


# Verifying Tactics: Analysis

In [22]:
# Type 1: Djokovic backhand down the line -> Nadal answer
djoko_bh_dtl = shots[(shots.hitter == "Novak Djokovic") &
                     (shots.stroke == "b") &
                     (shots.direction == 1)].copy()
djoko_bh_dtl["nextIndex"] = djoko_bh_dtl["shotIndex"] + 1

djokoTactic = djoko_bh_dtl.merge(
    shots[["Pt", "shotIndex", "targetZone", "stroke", "direction"]],
    left_on=["Pt", "nextIndex"],
    right_on=["Pt", "shotIndex"],
    suffixes=("_djoko", "_nadal")
).drop(columns="nextIndex")

djokoTactic

,Pt,shotIndex_djoko,server,hitter,stroke_djoko,positions,direction_djoko,errorCode,outcome,targetZone_djoko,shotIndex_nadal,targetZone_nadal,stroke_nadal,direction_nadal
0,5,1,Novak Djokovic,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,2,to opponent BH,b,3
1,11,9,Novak Djokovic,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,10,to opponent BH,s,3
2,19,0,Rafael Nadal,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,1,middle,f,2
3,19,12,Rafael Nadal,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,13,to opponent BH,s,3
4,23,3,Novak Djokovic,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,4,to opponent BH,s,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,354,10,Rafael Nadal,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,11,to opponent FH,b,1
91,362,1,Novak Djokovic,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,2,to opponent FH,b,1
92,362,15,Novak Djokovic,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,16,middle,b,2
93,365,3,Novak Djokovic,Novak Djokovic,b,NaN,1,NaN,NaN,to opponent BH,4,middle,s,2


In [24]:
djokoTactic["prevIndex"] = djokoTactic["shotIndex_djoko"] - 1
djokoTactic = djokoTactic.merge(
    shots[["Pt", "shotIndex", "targetZone"]],
    left_on=["Pt", "prevIndex"],
    right_on=["Pt", "shotIndex"],
    how="left",
    suffixes=("", "_prev")
).drop(columns=["prevIndex", "shotIndex"]).rename(columns={"targetZone": "targetZone_nadal_prev"})

fixed = ["stroke_djoko", "direction_djoko", "stroke_nadal", "direction_nadal", "targetZone_nadal_prev", "targetZone_djoko", "targetZone_nadal"]
rest = [col for col in djokoTactic.columns if col not in fixed]
# rest is now [X, Y]
djokoTactic = djokoTactic[rest + fixed]

djokoTactic

,Pt,shotIndex_djoko,server,hitter,positions,errorCode,outcome,shotIndex_nadal,stroke_djoko,direction_djoko,stroke_nadal,direction_nadal,targetZone_nadal_prev,targetZone_nadal_prev,targetZone_djoko,targetZone_nadal
0,5,1,Novak Djokovic,Novak Djokovic,NaN,NaN,NaN,2,b,1,b,3,to opponent BH,to opponent BH,to opponent BH,to opponent BH
1,11,9,Novak Djokovic,Novak Djokovic,NaN,NaN,NaN,10,b,1,s,3,to opponent BH,to opponent BH,to opponent BH,to opponent BH
2,19,0,Rafael Nadal,Novak Djokovic,NaN,NaN,NaN,1,b,1,f,2,NaN,NaN,to opponent BH,middle
3,19,12,Rafael Nadal,Novak Djokovic,NaN,NaN,NaN,13,b,1,s,3,middle,middle,to opponent BH,to opponent BH
4,23,3,Novak Djokovic,Novak Djokovic,NaN,NaN,NaN,4,b,1,s,3,middle,middle,to opponent BH,to opponent BH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,354,10,Rafael Nadal,Novak Djokovic,NaN,NaN,NaN,11,b,1,b,1,middle,middle,to opponent BH,to opponent FH
91,362,1,Novak Djokovic,Novak Djokovic,NaN,NaN,NaN,2,b,1,b,1,middle,middle,to opponent BH,to opponent FH
92,362,15,Novak Djokovic,Novak Djokovic,NaN,NaN,NaN,16,b,1,b,2,middle,middle,to opponent BH,middle
93,365,3,Novak Djokovic,Novak Djokovic,NaN,NaN,NaN,4,b,1,s,2,to opponent BH,to opponent BH,to opponent BH,middle


In [31]:
djokoTactic["stroke_nadal"].value_counts()

stroke_nadal
s    43
b    37
f     4
d     4
n     3
w     3
r     1
Name: count, dtype: int64

In [34]:
djokoNadalSlice = djokoTactic[djokoTactic["stroke_nadal"] == 's']
djokoNadalBH = djokoTactic[djokoTactic["stroke_nadal"] == 'b']
djokoNadalRest = djokoTactic[~djokoTactic["stroke_nadal"].isin(['s', 'b'])]

print(df[df.index.isin(djokoNadalSlice['Pt'].unique())][["PtWinner"]].value_counts())
print(df[df.index.isin(djokoNadalBH['Pt'].unique())][["PtWinner"]].value_counts())
print(df[df.index.isin(djokoNadalRest['Pt'].unique())][["PtWinner"]].value_counts())

PtWinner
1           21
2           17
Name: count, dtype: int64
PtWinner
1           19
2           16
Name: count, dtype: int64
PtWinner
2           13
1            2
Name: count, dtype: int64


In [21]:
djokoBHDtlPoints = df[df.index.isin(djokoTactic['Pt'].unique())][["PtWinner"]]
djokoBHDtlPoints["PtWinner"].value_counts()

PtWinner
1    38
2    37
Name: count, dtype: int64

In [9]:
# Type 1: Djokovic backhand down the line -> Nadal answer
djoko_bh_dtl = shots[(shots.hitter == "Novak Djokovic") &
                     (shots.stroke == "b") &
                     (shots.direction == 1)].copy()
djoko_bh_dtl["prevIndex"] = djoko_bh_dtl["shotIndex"] - 1
djoko_bh_dtl["nextIndex"] = djoko_bh_dtl["shotIndex"] + 1

responses_type1 = djoko_bh_dtl.merge(
    shots,
    left_on=["Pt", "nextIndex"],
    right_on=["Pt", "shotIndex"],
    suffixes=("_djoko", "_nadal")
)
responses_type1 = responses_type1[responses_type1["hitter"] == "Rafael Nadal"]

# Nadal’s reply direction & court position
resp_dir_counts = responses_type1["targetZone_nadal"].value_counts()
resp_position_counts = responses_type1["positions_nadal"].value_counts()
pts_type1 = sorted(responses_type1["Pt"].unique())


# Djokovic backhand down the line -> Nadal response
resp_dir_counts = (
    responses_type1["targetZone_nadal"]
    .value_counts(dropna=False)
    .rename_axis("direction")
    .to_frame("count")
)
resp_dir_counts["pct"] = resp_dir_counts["count"] / resp_dir_counts["count"].sum() * 100

KeyError: 'hitter'

In [ ]:
# Type 2: Nadal FH inside-out (direction 1) → Djokovic’s reply
nadal_inside_out = shots[(shots.hitter == "Rafael Nadal") &
                         (shots.stroke == "f") &
                         (shots.direction == 1)].copy()
nadal_inside_out["nextIndex"] = nadal_inside_out["shotIndex"] + 1

responses_type2 = nadal_inside_out.merge(
    shots,
    left_on=["Pt", "nextIndex"],
    right_on=["Pt", "shotIndex"],
    suffixes=("_nadal", "_djoko")
)
responses_type2 = responses_type2[responses_type2["hitter"] == "Novak Djokovic"]

djoko_resp_dir_counts = responses_type2["targetZone_djoko"].value_counts()
djoko_resp_position_counts = responses_type2["positions_djoko"].value_counts()
pts_type2 = sorted(responses_type2["Pt"].unique())


# Nadal FH inside-out → Djokovic response
djoko_resp_dir_counts = (
    responses_type2["targetZone_djoko"]
    .value_counts(dropna=False)
    .rename_axis("direction")
    .to_frame("count")
)
djoko_resp_dir_counts["pct"] = (
    djoko_resp_dir_counts["count"] / djoko_resp_dir_counts["count"].sum() * 100
)